# 02 Baseline Forecasting - NESO Daily Demand

This notebook establishes clean baseline forecasting performance before any SARIMA, SARIMAX, Prophet, machine learning, deep learning or simulation work is attempted.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src import baseline_models
from src.utils import FIGURES_DIR, TABLES_DIR

TARGET = "nd_mean"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

## Load Processed Daily Data

The modelling phase uses `data/processed/daily_demand_2019_2025.csv`, generated by `python src/prepare_data.py`.

In [ ]:
daily_df = baseline_models.load_daily_dataset()
print("Shape:", daily_df.shape)
display(daily_df.head())
display(daily_df.tail())

## Target Summary

In [ ]:
series = baseline_models.create_time_series(daily_df, target=TARGET)
train, test = baseline_models.train_test_split_time_series(series, TEST_START, TEST_END)

print("Target:", TARGET)
print("Train range:", train.index.min(), "to", train.index.max(), "rows:", len(train))
print("Test range:", test.index.min(), "to", test.index.max(), "rows:", len(test))
display(series.describe().to_frame(TARGET))

ax = series.plot(figsize=(14, 5), title=f"Daily demand target: {TARGET}")
ax.axvline(pd.Timestamp(TEST_START), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("date")
ax.set_ylabel(TARGET)
plt.show()

## Run Baseline Forecasting Pipeline

The pipeline compares last-value naive, previous-year seasonal naive and Holt-Winters Exponential Smoothing baselines.

In [ ]:
comparison, forecasts = baseline_models.run_baseline_forecasting(
    target=TARGET,
    test_start=TEST_START,
    test_end=TEST_END,
)
display(comparison)
display(forecasts.head())

## Forecast Plots

In [ ]:
figures = [
    FIGURES_DIR / "modelling" / "actual_vs_baseline_forecasts.png",
    FIGURES_DIR / "modelling" / "forecast_errors_by_model.png",
]
for figure_path in figures:
    print(figure_path)
    if figure_path.exists():
        image = plt.imread(figure_path)
        plt.figure(figsize=(14, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

## Baseline Interpretation

The best baseline is the model with the lowest holdout MAE, with RMSE and MAPE used as supporting diagnostics. Baselines are needed before SARIMA, SARIMAX, Prophet or more complex approaches because they define the minimum level of performance that later models must beat. If an advanced model cannot improve on a simple seasonal naive or Holt-Winters baseline, its extra complexity is not justified.

This notebook does not build SARIMAX, Prophet, machine learning, deep learning or simulation models. Those belong in later phases after baseline performance has been reviewed.